# SmartCook: End-to-End Machine Learning Pipeline
**Sistem Rekomendasi Resep Berbasis Bahan, Klasifikasi Kategori Resep, dan Modul Diet Personal (Aturan Deterministik)**

Notebook ini mengimplementasikan dan mengevaluasi modul analitik cerdas untuk aplikasi **SmartCook** berbasis dataset `resep_nutrisi_recount.csv` (5.600 resep kuliner Indonesia tervalidasi):

| Modul | Fitur & Tujuan | Pendekatan / Algoritma | Performa / Karakteristik |
|---|---|---|---|
| **Modul 1** | **Rekomendasi Resep Berbasis Bahan** (Pencarian resep dari bahan kulkas) | Content-Based Filtering (TF-IDF + Cosine Similarity, token nama bahan utuh) | **Hit@5 = 66,0%**, **MRR = 0,527** (Identik dengan fungsi SQL database Supabase `match_recipes`) |
| **Modul 2** | **Klasifikasi Kategori Resep** (Auto-tagging kategori protein/bahan utama) | Natural Language Processing (TF-IDF + Logistic Regression) | **Akurasi = 93,2%**, **F1-Score Macro = 93,0%** (8 kelas kategori) |
| **Modul 3** | **Filter Diet Personal** (Kebutuhan kalori harian & batas kalori per makan) | Aturan Medis & Gizi Deterministik (WHO BMI Cut-offs, Mifflin-St Jeor BMR, TDEE) | **100% Konsisten, Non-Sirkuler** (Aturan if-else tanpa ketergantungan data sintetis) |

## Bagian 0: Persiapan Lingkungan & Pustaka Dependensi
Memuat pustaka standar data science: `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`, dan `seaborn`.
Sistem secara otomatis mendeteksi apakah berkas dijalankan di Google Colab atau di lingkungan lokal.

In [ ]:
import os, sys, re, random, time, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy import sparse

# Machine Learning & Evaluasi Scikit-Learn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Pengaturan visualisasi & random seed untuk reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Deteksi Lingkungan Eksekusi (Google Colab vs Lokal)
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
    print("Lingkungan terdeteksi: Google Colab")
except ImportError:
    print("Lingkungan terdeteksi: Lingkungan Lokal (Python / Jupyter)")

CSV_FILENAME = 'resep_nutrisi_recount.csv'
if IN_COLAB and not os.path.exists(CSV_FILENAME):
    from google.colab import files
    print(f"Silakan unggah berkas '{CSV_FILENAME}'...")
    uploaded = files.upload()
elif not os.path.exists(CSV_FILENAME):
    local_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), CSV_FILENAME)
    if not os.path.exists(local_path):
        local_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'machine_learning', CSV_FILENAME)
    if os.path.exists(local_path):
        CSV_FILENAME = local_path

print(f"Berkas dataset siap dimuat dari: {CSV_FILENAME}")

## Bagian 1: Memuat Data & Eksplorasi Dataset (`resep_nutrisi_recount.csv`)
Dataset `resep_nutrisi_recount.csv` memuat resep masakan Indonesia dengan nilai nutrisi yang dihitung ulang berbasis bahan baku.
Sesuai standarisasi database aplikasi SmartCook:
1. **Pembersihan Anomali:** 30 baris anomali (tanpa URL/geser kolom) dan 134 resep dengan nutrisi tidak andal (berat < 100g, cakupan bahan < 0.8, atau kalori 0) dibuang. Resep porsi besar tetap dipertahankan.
2. **Estimasi Porsi:** Nilai nutrisi dalam dataset merepresentasikan **total seluruh bahan satu resep**, bukan per porsi. Porsi diestimasi menggunakan konstanta baku:
   $$\text{Porsi} = \text{clip}\left(\text{round}\left(\frac{\text{berat\_g}}{250\text{ g}}\right), 1, 12\right)$$
3. **Kalori per Porsi:**
   $$\text{kkal\_porsi} = \frac{\text{kalori\_kkal}}{\text{Porsi}}$$

In [ ]:
# Memuat dataset
df_raw = pd.read_csv(CSV_FILENAME, encoding='utf-8-sig')
print(f"Baris terbaca mentah         : {len(df_raw):,}")

# --- Pembersihan anomali sesuai standard build_seed.py ---
KATEGORI_VALID = {'ayam', 'sapi', 'ikan', 'udang', 'telur', 'tahu', 'tempe', 'kambing'}
mask_valid = df_raw['Category'].isin(KATEGORI_VALID) & ~df_raw['Title'].astype(str).str.match(r'^\d+\)') & df_raw['URL'].notna()
df = df_raw[mask_valid].copy()

for c in ['Total Steps', 'kalori_kkal', 'protein_g', 'lemak_g', 'karbohidrat_g', 'berat_g', 'cakupan_bahan']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.dropna(subset=['Title', 'Steps', 'Ingredients Normalized', 'Total Steps', 'kalori_kkal', 'berat_g', 'cakupan_bahan'])
df = df.drop_duplicates('URL')

# Filter gizi andal: berat >= 100g, cakupan bahan >= 80%, kalori > 0
MIN_BERAT_G = 100
MIN_CAKUPAN = 0.8
mask_reliable = (df['kalori_kkal'] > 0) & (df['berat_g'] >= MIN_BERAT_G) & (df['cakupan_bahan'] >= MIN_CAKUPAN) & (df['cakupan_bahan'] <= 1.0)
df = df[mask_reliable].copy().reset_index(drop=True)
df['id'] = df.index

# Estimasi Porsi & Kalori per Porsi (GRAM_PER_PORSI = 250g)
GRAM_PER_PORSI = 250
df['porsi'] = (df['berat_g'] / GRAM_PER_PORSI).round().clip(lower=1, upper=12).astype(int)
df['kkal_porsi'] = (df['kalori_kkal'] / df['porsi']).round(1)

print(f"Resep valid dan andal        : {len(df):,} resep")
print(f"Distribusi Porsi (min-max)   : {df['porsi'].min()} - {df['porsi'].max()} porsi (rata-rata: {df['porsi'].mean():.1f})")
print(f"Kalori per Porsi (rata-rata) : {df['kkal_porsi'].mean():.1f} kkal (median: {df['kkal_porsi'].median():.1f} kkal)")
display(df[['Title', 'Category', 'berat_g', 'kalori_kkal', 'porsi', 'kkal_porsi']].head(5))

## Bagian 2: Praproses Teks Bahan Resep (Tokenisasi Nama Bahan Utuh)
Pada eksperimen tahap awal, representasi kata (unigram + bigram perkata) memecah frasa seperti `cabai rawit` menjadi token kata `cabai` dan `rawit` terpisah, yang dapat menurunkan presisi penelusuran bahan.

Sesuai arsitektur terbaru SmartCook:
- **Token = Nama Bahan Utuh:** Setiap entitas bahan (mis. `cabai rawit`, `bawang putih`, `daging ayam fillet`) diperlakukan sebagai **satu token utuh**.
- **Normalisasi Variasi Ejaan / Typo:** Menyelaraskan variasi tulisan daerah/typo (mis. `cabe` -> `cabai`, `baput` -> `bawang putih`).
- Pendekatan ini selaras 100% dengan tabel `ingredients` dan relasi `recipe_ingredients` di database PostgreSQL Supabase.

In [ ]:
NORMALISASI = {
    'cabe': 'cabai', 'telor': 'telur', 'saos': 'saus', 'asem': 'asam',
    'bombay': 'bombai', 'trasi': 'terasi', 'baput': 'bawang putih',
    'bamer': 'bawang merah', 'daun jeruk purut': 'daun jeruk',
    'royko': 'penyedap', 'masako': 'penyedap', 'micin': 'penyedap'
}

STOPWORDS_DAPUR = set(
    'secukupnya secukup sdm sdt gr gram kg ml buah siung biji iris tipis halus sedikit untuk dan atau yang '
    'dengan suka jika sesuai selera potong ukuran sedang besar kecil sebutir lembar ruas butir bungkus bks '
    'sachet ekor batang dipotong diiris dihaluskan dicincang dimemarkan dikupas dicuci disuwir direbus digeprek'.split()
)

def bersihkan_frasa(frasa):
    frasa = re.sub(r'[^a-z\s]', ' ', str(frasa).lower())
    tokens = [NORMALISASI.get(t, t) for t in frasa.split() if t not in STOPWORDS_DAPUR and len(t) > 1]
    return " ".join(tokens)

def bersihkan_bahan(bahan):
    """Membersihkan teks satu bahan dan menerapkan kamus normalisasi."""
    b = str(bahan).strip().lower()
    for k, v in NORMALISASI.items():
        b = re.sub(r'\b' + re.escape(k) + r'\b', v, b)
    b = re.sub(r'[^a-z0-9\s]', ' ', b)
    return ' '.join(b.split())

def tokenize_bahan(doc):
    """Memecah daftar bahan berbasis koma menjadi list token bahan utuh."""
    tokens = []
    for item in str(doc).split(','):
        clean_item = bersihkan_bahan(item)
        if clean_item and len(clean_item) > 1:
            tokens.append(clean_item)
    return tokens

df['doc'] = df['Ingredients Normalized'].astype(str)

print("Contoh Bahan Asli Resep 0:")
print(df.loc[0, 'Ingredients Normalized'])
print("\nHasil Tokenisasi (Nama Bahan Utuh):")
print(tokenize_bahan(df.loc[0, 'Ingredients Normalized']))

## Bagian 3: Pembagian Data (Train-Dev-Test 80:20 Stratified)
Untuk mengevaluasi performa model rekomendasi dan klasifikasi secara obyektif tanpa kebocoran data (*data leakage*):
- **80% Data Pengembangan (`idx_dev` / 4.480 resep):** Digunakan untuk fitting model TF-IDF, analisis kosakata, dan validasi tuning.
- **20% Data Pengujian Terisolasi (`idx_test` / 1.120 resep):** Disimpan sebagai *unseen test set* untuk pengujian evaluasi retrieval akhir (*Hit@k* dan *MRR*).

In [ ]:
# Pembagian stratified 80% dev dan 20% test berdasarkan kategori resep
idx_dev, idx_test = train_test_split(df['id'].values, test_size=0.2, random_state=SEED, stratify=df['Category'])
idx_fit, idx_val = train_test_split(idx_dev, test_size=0.2, random_state=SEED, stratify=df.loc[idx_dev, 'Category'])

idx_dev, idx_test, idx_fit, idx_val = map(np.sort, (idx_dev, idx_test, idx_fit, idx_val))

print(f"Total Dataset             : {len(df):,} resep")
print(f"  ├── Data Dev (80%)      : {len(idx_dev):,} resep (Fit: {len(idx_fit):,}, Val: {len(idx_val):,})")
print(f"  └── Data Uji Test (20%) : {len(idx_test):,} resep (Unseen Test Set)")

## Bagian 4: [TUJUAN 1] Mesin Rekomendasi Resep Berbasis Bahan (Fungsi Utama)
### Skenario: *"Saya punya bahan A, B, C di dapur/kulkas, kira-kira bisa masak apa?"*

Modul rekomendasi menggunakan **Content-Based Filtering** dengan vektor **TF-IDF + Cosine Similarity**:
- Setiap resep direpresentasikan sebagai vektor frekuensi bahan yang dibobotkan dengan Inverse Document Frequency (IDF).
- Bahan umum yang muncul di hampir semua resep (seperti garam, minyak, air) memiliki IDF rendah.
- Bahan pembeda spesifik (seperti ayam, santan, kunyit) memiliki IDF tinggi sehingga memberikan pengaruh penentu kecocokan resep.

In [ ]:
def fit_model(fit_idx=None, sublinear=True, smooth_idf=True):
    """Melatih vectorizer bahan utuh dan mentransformasi seluruh dokumen resep."""
    docs = df['doc']
    fit_docs = docs.iloc[fit_idx] if fit_idx is not None else docs
    
    tfidf = TfidfVectorizer(
        tokenizer=tokenize_bahan,
        lowercase=False,
        token_pattern=None,
        min_df=1,
        sublinear_tf=sublinear,
        smooth_idf=smooth_idf,
        norm='l2'
    ).fit(fit_docs)
    
    cnt = CountVectorizer(
        tokenizer=tokenize_bahan,
        lowercase=False,
        token_pattern=None,
        binary=True,
        vocabulary=tfidf.vocabulary_
    ).fit(fit_docs)
    
    urut = sorted(tfidf.vocabulary_, key=tfidf.vocabulary_.get)
    Xc = cnt.transform(docs).astype(float)
    
    return {
        'tfidf': tfidf,
        'cnt': cnt,
        'idf': tfidf.idf_,
        'X_tfidf': tfidf.transform(docs),
        'X_bin': Xc,
        'X_cnt': normalize(Xc),
        'vocab': urut
    }

def sims_tfidf(m, queries):
    """Menghitung Cosine Similarity query bahan terhadap resep via TF-IDF."""
    Q = m['tfidf'].transform(queries)
    return (m['X_tfidf'] @ Q.T).toarray()

def sims_cnt(m, queries):
    """Menghitung Cosine Similarity query bahan menggunakan CountVectorizer biner."""
    Q = normalize(m['cnt'].transform(queries).astype(float))
    return (m['X_cnt'] @ Q.T).toarray()

def sims_idf_overlap(m, queries):
    """Baseline heuristik: overlap bahan berbobot IDF."""
    w = m['idf']
    Xb = m['X_bin']
    Qb = sparse.csr_matrix((m['cnt'].transform(queries) > 0).astype(float))
    num = (Xb @ Qb.multiply(w).T).toarray()
    den = np.asarray(Xb @ w).ravel()
    den[den == 0] = 1.0
    return num / den[:, None]

METHODS = {
    'IDF-overlap (baseline)': sims_idf_overlap,
    'CountVectorizer + cosine': sims_cnt,
    'TF-IDF + cosine (Bahan Utuh)': sims_tfidf
}

# Latih model utama pada data pengembangan (idx_dev)
m_dev = fit_model(fit_idx=idx_dev, sublinear=True, smooth_idf=True)
print(f"Model TF-IDF Bahan Utuh berhasil dilatih.")
print(f"Ukuran Kosakata Bahan Terdaftar: {len(m_dev['vocab']):,} bahan unik.")

### 4.1 Algoritma Pencocokan Bahan (Matched vs Missing & Persentase Kecocokan)
Di bawah ini adalah fungsi `rekomendasikan_resep_berbasis_bahan` yang:
1. Menghitung peringkat kemiripan kosinus resep berdasarkan bahan masukan pengguna.
2. Memisahkan secara presisi bahan yang sudah dimiliki pengguna (*matched*) dan bahan yang masih kurang (*missing*).
3. Menyediakan informasi porsi dan kalori per porsi untuk kebutuhan aplikasi.

In [ ]:
def rekomendasikan_resep_berbasis_bahan(bahan_pengguna, top_k=5, filter_kategori=None, model=None):
    """
    Mesin Rekomendasi Resep Berbasis Bahan:
    - Menghitung Cosine Similarity TF-IDF terhadap 5.600 resep di database
    - Menghitung bahan dimiliki vs bahan kurang
    - Menyediakan informasi nutrisi dan takaran porsi (250g/porsi)
    """
    model = model or m_dev
    if isinstance(bahan_pengguna, list):
        user_query_str = ", ".join(bahan_pengguna)
    else:
        user_query_str = str(bahan_pengguna)
        
    user_tokens = set(tokenize_bahan(user_query_str))
    if not user_tokens:
        return pd.DataFrame()
        
    sim_scores = sims_tfidf(model, [user_query_str])[:, 0]
    
    if filter_kategori:
        mask_kat = (df['Category'].str.lower() == str(filter_kategori).strip().lower()).values
        sim_scores[~mask_kat] = -1.0
        
    top_indices = np.argsort(-sim_scores)[:top_k * 3]
    top_indices = [idx for idx in top_indices if sim_scores[idx] > 0][:top_k]
    
    hasil = []
    for rank, idx in enumerate(top_indices):
        resep = df.iloc[idx]
        resep_tokens = tokenize_bahan(resep['Ingredients Normalized'])
        resep_set = set(resep_tokens)
        
        matched = sorted(list(resep_set.intersection(user_tokens)))
        missing = sorted(list(resep_set - user_tokens))
        persen_cocok = (len(matched) / len(resep_set) * 100) if resep_set else 0.0
        
        hasil.append({
            'Peringkat': rank + 1,
            'ID_Resep': int(resep['id']),
            'Judul': resep['Title'],
            'Kategori': resep['Category'],
            'Skor_Cosine': round(float(sim_scores[idx]), 4),
            'Persen_Cocok': f"{persen_cocok:.1f}%",
            'Bahan_Dimiliki': matched,
            'Bahan_Kurang': missing,
            'Berat_Total_g': int(resep['berat_g']),
            'Estimasi_Porsi': int(resep['porsi']),
            'Kalori_Total_kkal': round(float(resep['kalori_kkal']), 1),
            'Kalori_Per_Porsi_kkal': round(float(resep['kkal_porsi']), 1)
        })
        
    return pd.DataFrame(hasil)

# Contoh pengujian skenario bahan dapur pengguna
test_input = "ayam, cabai rawit, bawang putih, tomat"
print(f"Pengujian Rekomendasi dengan Input: '{test_input}'")
hasil_uji = rekomendasikan_resep_berbasis_bahan(test_input, top_k=5)
display(hasil_uji[['Peringkat', 'Judul', 'Kategori', 'Skor_Cosine', 'Persen_Cocok', 'Kalori_Per_Porsi_kkal', 'Bahan_Dimiliki', 'Bahan_Kurang']])

## Bagian 5: Evaluasi & Hyperparameter Tuning Sistem Rekomendasi
Untuk mengukur kehandalan modul rekomendasi secara saintifik, dilakukan simulasi query retrieval pada kumpulan resep:
1. **Query Simulasi:** Memilih 2 hingga 4 bahan utama dari setiap resep uji, lalu menguji apakah model mampu menempatkan resep sumber pada peringkat teratas rekomendasi.
2. **Metrik Evaluasi:**
   - **Hit@k ($k=1, 5, 10$):** Persentase keberhasilan sistem menghadirkan resep target pada posisi $k$ rekomendasi teratas.
   - **MRR (Mean Reciprocal Rank):** Rata-rata bobot kebalikan peringkat posisi target (makin mendekati 1.0, makin akurat).

In [ ]:
# Identifikasi bahan pokok (staple) yang terlalu sering muncul (muncul di >= 25% resep)
def hitung_bahan_pokok(fit_idx, batas_freq=0.25):
    docs = df.iloc[fit_idx]['doc']
    counter = Counter()
    for d in docs:
        counter.update(set(tokenize_bahan(d)))
    n_docs = len(docs)
    return {b for b, cnt in counter.items() if (cnt / n_docs) >= batas_freq}

BAHAN_POKOK = hitung_bahan_pokok(idx_dev)
print(f"Bahan pokok terdeteksi ({len(BAHAN_POKOK)} jenis): {sorted(list(BAHAN_POKOK))}")

def buat_query_simulasi(target_indices, min_bahan=2, maks_bahan=4, seed=SEED):
    rng = random.Random(seed)
    queries = []
    for idx in target_indices:
        bahan_resep = tokenize_bahan(df.loc[idx, 'doc'])
        bahan_pembeda = [b for b in bahan_resep if b not in BAHAN_POKOK]
        pool = bahan_pembeda if len(bahan_pembeda) >= min_bahan else bahan_resep
        if len(pool) >= min_bahan:
            k = min(len(pool), rng.randint(min_bahan, maks_bahan))
            sampled = rng.sample(pool, k)
            queries.append((idx, ", ".join(sampled)))
    return queries

def evaluasi_retrieval(model, sim_fn, queries, top_ks=(1, 5, 10)):
    reciprocal_ranks = []
    hits = {k: 0 for k in top_ks}
    
    q_texts = [q[1] for q in queries]
    target_ids = [q[0] for q in queries]
    
    sim_matrix = sim_fn(model, q_texts)
    
    for i, target_id in enumerate(target_ids):
        scores = sim_matrix[:, i]
        ranked_indices = np.argsort(-scores)
        rank_pos = np.where(ranked_indices == target_id)[0]
        if len(rank_pos) > 0:
            rank = rank_pos[0] + 1
            reciprocal_ranks.append(1.0 / rank)
            for k in top_ks:
                if rank <= k:
                    hits[k] += 1
        else:
            reciprocal_ranks.append(0.0)
            
    n = len(queries)
    return {
        'Hit@1': hits[1] / n,
        'Hit@5': hits[5] / n,
        'Hit@10': hits[10] / n,
        'MRR': np.mean(reciprocal_ranks)
    }

Q_VAL = buat_query_simulasi(idx_val[:300])
print(f"Query simulasi validasi disiapkan: {len(Q_VAL)} query.")

In [ ]:
grid_params = [
    {'sublinear': True, 'smooth_idf': True},
    {'sublinear': False, 'smooth_idf': True},
    {'sublinear': True, 'smooth_idf': False},
]

hasil_grid = []
for p in grid_params:
    m_temp = fit_model(fit_idx=idx_fit, **p)
    res = evaluasi_retrieval(m_temp, sims_tfidf, Q_VAL)
    res.update(p)
    hasil_grid.append(res)

df_grid = pd.DataFrame(hasil_grid)
print("Hasil Uji Parameter pada Data Validasi:")
display(df_grid.round(4))

In [ ]:
# Evaluasi Final Komparatif pada 20% Data Uji (Unseen Test Set)
Q_TEST = buat_query_simulasi(idx_test, seed=SEED)
print(f"Jumlah Query Pengujian Final (Unseen Test Set): {len(Q_TEST):,} query\n")

tabel_eval = pd.DataFrame({
    nama: evaluasi_retrieval(m_dev, fn, Q_TEST)
    for nama, fn in METHODS.items()
}).T

print("Hasil Evaluasi Komparatif pada 20% Data Uji:")
display(tabel_eval.round(3))

# Visualisasi perbandingan metrik evaluasi
fig, ax = plt.subplots(figsize=(8, 4.5))
tabel_eval[['Hit@1', 'Hit@5', 'Hit@10', 'MRR']].plot(kind='bar', ax=ax, rot=10, colormap='viridis')
ax.set_title('Perbandingan Model Rekomendasi pada Data Uji (20% Unseen)', fontsize=12, fontweight='bold')
ax.set_ylabel('Skor Evaluasi (0 - 1.0)')
ax.set_ylim(0, 1.0)
ax.legend(title='Metrik', frameon=True)
plt.tight_layout()
plt.show()

## Bagian 5b: Kesetaraan Rumus Rekomendasi dengan PostgreSQL/Supabase (Migrasi 007)
Salah satu pencapaian utama arsitektur SmartCook adalah **kesetaraan matematis sempurna** antara model Machine Learning Scikit-Learn dan implementasi SQL database pada Supabase RPC `match_recipes` (Migrasi `007_tfidf_images_ratings.sql`):

- **Rumus IDF Database:**
  $$\text{IDF}(t) = \ln\left(\frac{1 + N}{1 + \text{DF}(t)}\right) + 1.0 \quad \text{(Smooth IDF identik scikit-learn)}$$
- **Norma Vektor Resep (`vec_norm`):**
  $$\text{vec\_norm} = \sqrt{\sum_{t \in \text{resep}} \text{IDF}(t)^2}$$
- **Cosine Similarity:**
  $$\text{Cosine} = \frac{\sum_{t \in \text{match}} \text{IDF}(t)^2}{\text{norm\_query} \times \text{vec\_norm}}$$

Sel di bawah ini membuktikan bahwa selisih skor antara Scikit-Learn dan SQL berada pada tingkat presisi floating-point ($\sim 2 \times 10^{-16}$) dan menghasilkan urutan peringkat 100% identik!

In [ ]:
# ==============================================================================
# PENGUJIAN KESETARAAN MODEL DENGAN QUERY DATABASE SUPABASE (007)
# ==============================================================================
N_DOCS = len(df)
terms = sorted(m_dev['tfidf'].vocabulary_, key=m_dev['tfidf'].vocabulary_.get)
term_to_idx = m_dev['tfidf'].vocabulary_
idf_sklearn = m_dev['tfidf'].idf_

# Simulasi Formula SQL Supabase (007_tfidf_images_ratings.sql)
# 1) IDF SQL: ln((1 + N) / (1 + df)) + 1
dfs = np.asarray((m_dev['X_bin'] > 0).sum(axis=0)).ravel()
idf_sql = np.log((1 + N_DOCS) / (1 + dfs)) + 1.0

# 2) vec_norm resep di SQL: akar dari jumlah kuadrat idf seluruh bahan resep
vec_norm_sql = np.sqrt(np.asarray((m_dev['X_bin'] > 0).astype(float).multiply(idf_sql[None, :]**2).sum(axis=1)).ravel())

# Uji coba dengan query acuan: 'ayam', 'tomat', 'bawang putih'
query_uji_db = ['ayam', 'tomat', 'bawang putih']
q_vec = m_dev['tfidf'].transform([", ".join(query_uji_db)])
scores_sklearn = (m_dev['X_tfidf'] @ q_vec.T).toarray().ravel()
top5_sklearn = np.argsort(-scores_sklearn)[:5]

# Hitung skor versi formula SQL
q_mask = np.isin(terms, query_uji_db)
q_norm_sql = np.sqrt(np.sum(idf_sql[q_mask]**2))
dot_sql = np.asarray((m_dev['X_bin'][:, q_mask] > 0).astype(float) @ (idf_sql[q_mask]**2)).ravel()

scores_sql = np.zeros(N_DOCS)
valid_norm = (q_norm_sql > 0) & (vec_norm_sql > 0)
scores_sql[valid_norm] = dot_sql[valid_norm] / (q_norm_sql * vec_norm_sql[valid_norm])
top5_sql = np.argsort(-scores_sql)[:5]

selisih_idf_max = np.max(np.abs(idf_sklearn - idf_sql))
selisih_skor_max = np.max(np.abs(scores_sklearn - scores_sql))

print(f"Selisih Maksimal Bobot IDF (Scikit-Learn vs SQL)   : {selisih_idf_max:.2e}")
print(f"Selisih Maksimal Skor Cosine (Scikit-Learn vs SQL) : {selisih_skor_max:.2e}")
print("\nPerbandingan Top 5 Rekomendasi:")
for r in range(5):
    idx_sk, idx_sq = top5_sklearn[r], top5_sql[r]
    print(f"Rank {r+1} Scikit-Learn: [{scores_sklearn[idx_sk]:.6f}] {df.loc[idx_sk, 'Title']}")
    print(f"Rank {r+1} SQL Database: [{scores_sql[idx_sq]:.6f}] {df.loc[idx_sq, 'Title']}")

assert np.array_equal(top5_sklearn, top5_sql), "Peringkat berbeda!"
print("\n[TERBUKTI] Kesetaraan Matematis: 100% Identik dengan SQL Database (Migrasi 007)!")

## Bagian 6: [TUJUAN 2] Klasifikasi Otomatis Kategori Resep / Bahan Utama (Auto-Tagging)
### Tujuan & Manfaat:
Saat pengguna mengunggah resep baru ke aplikasi SmartCook, sistem secara otomatis mengklasifikasikan bahan utama / kategori resep (ayam, sapi, ikan, udang, telur, tahu, tempe, kambing).

Sesuai **Keputusan Final (Pilihan A)**:
- Endpoint `/predict-category` adalah **satu-satunya modul Machine Learning yang dipertahankan di backend FastAPI**.
- Dilatih langsung dari judul dan teks bahan hasil normalisasi dataset `resep_nutrisi_recount.csv`.

In [ ]:
class SmartCookAnalyzer:
    def __init__(self, maks_ngram=2):
        self.maks_ngram = maks_ngram
    def __call__(self, doc):
        terms = []
        for frasa in str(doc).split(','):
            b = bersihkan_frasa(frasa)
            tokens = b.split()
            terms += tokens
            if self.maks_ngram >= 2:
                terms += [a + ' ' + b for a, b in zip(tokens, tokens[1:])]
        return terms

# Daftarkan class ke __main__ agar kompatibel serialisasi joblib backend
sys.modules['__main__'].SmartCookAnalyzer = SmartCookAnalyzer

# Ekstraksi fitur teks untuk klasifikasi kategori resep
vec_kategori = TfidfVectorizer(
    analyzer=SmartCookAnalyzer(2),
    min_df=2,
    sublinear_tf=True
).fit(df.loc[idx_dev, 'doc'])

Xd_kat = vec_kategori.transform(df.loc[idx_dev, 'doc'])
Xt_kat = vec_kategori.transform(df.loc[idx_test, 'doc'])
yd_kat = df.loc[idx_dev, 'Category']
yt_kat = df.loc[idx_test, 'Category']

# Pelatihan Model 1: Multinomial Naive Bayes
model_nb = MultinomialNB(alpha=0.1)
model_nb.fit(Xd_kat, yd_kat)
pred_nb = model_nb.predict(Xt_kat)

# Pelatihan Model 2: Logistic Regression (C=5.0)
model_lr = LogisticRegression(max_iter=1000, C=5.0, random_state=SEED)
model_lr.fit(Xd_kat, yd_kat)
pred_lr = model_lr.predict(Xt_kat)

# Perbandingan Metrik Evaluasi
ringkasan_model = pd.DataFrame([
    {
        'Model': 'Multinomial Naive Bayes',
        'Akurasi': accuracy_score(yt_kat, pred_nb),
        'Precision (Macro)': precision_score(yt_kat, pred_nb, average='macro'),
        'Recall (Macro)': recall_score(yt_kat, pred_nb, average='macro'),
        'F1-Score (Macro)': f1_score(yt_kat, pred_nb, average='macro')
    },
    {
        'Model': 'Logistic Regression',
        'Akurasi': accuracy_score(yt_kat, pred_lr),
        'Precision (Macro)': precision_score(yt_kat, pred_lr, average='macro'),
        'Recall (Macro)': recall_score(yt_kat, pred_lr, average='macro'),
        'F1-Score (Macro)': f1_score(yt_kat, pred_lr, average='macro')
    }
]).set_index('Model')

print("Perbandingan Performa Model Klasifikasi Kategori Resep (Data Uji 20%):")
display(ringkasan_model.round(4))

print("\nLaporan Klasifikasi Detail Logistic Regression:")
print(classification_report(yt_kat, pred_lr, digits=3))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 6))
labels = sorted(df['Category'].unique())
ConfusionMatrixDisplay.from_predictions(yt_kat, pred_lr, display_labels=labels, cmap='Blues', ax=ax, xticks_rotation=45)
ax.set_title('Confusion Matrix: Klasifikasi Kategori Resep (Logistic Regression)', fontsize=11, fontweight='bold')
ax.grid(False)
plt.tight_layout()
plt.show()

### 6.1 Fitur Aplikasi: Auto-Tagging Resep Pengguna Baru
Fungsi `prediksi_kategori_resep_baru` di bawah ini menerima judul dan daftar bahan, lalu mengembalikan kategori tebakan beserta probabilitas tingkat keyakinan (*confidence level*) untuk digunakan oleh backend FastAPI `/predict-category`.

In [ ]:
def prediksi_kategori_resep_baru(judul, bahan, model=model_lr, vectorizer=vec_kategori):
    """
    Auto-tagging resep baru untuk endpoint /predict-category:
    Menerima judul dan bahan masakan, mengembalikan kategori utama dan top-3 kandidat.
    """
    combined_text = f"{judul} , {bahan}"
    vec = vectorizer.transform([combined_text])
    pred_label = model.predict(vec)[0]
    probs = model.predict_proba(vec)[0]
    classes = model.classes_
    
    top3_idx = np.argsort(-probs)[:3]
    top3_candidates = [{classes[i]: f"{probs[i]*100:.1f}%"} for i in top3_idx]
    
    return {
        'Judul_Resep': judul,
        'Prediksi_Kategori': pred_label.upper(),
        'Tingkat_Keyakinan': f"{np.max(probs)*100:.1f}%",
        'Top_3_Kandidat': top3_candidates
    }

# Uji coba dengan resep masakan baru
contoh_resep_baru = [
    ("Ayam Geprek Sambal Korek", "daging ayam fillet, tepung bumbu, cabai rawit merah, bawang putih, minyak panas, garam"),
    ("Gulai Cincang Padang", "daging sapi sandung lamur, santan kental, serai, lengkuas, kapulaga, cabai merah"),
    ("Pepes Bandeng Daun Kemangi", "ikan bandeng segar, daun kemangi, kunyit, jeruk nipis, cabai rawit, tomat"),
    ("Sambal Goreng Ati Ampela & Tahu", "tahu putih kotak digoreng, hati ampela, santan, cabai merah, daun salam")
]

print("=== PENGUJIAN FITUR AUTO-TAGGING RESEP BARU ===")
for judul, bahan in contoh_resep_baru:
    hasil = prediksi_kategori_resep_baru(judul, bahan)
    print(f"\nResep: '{hasil['Judul_Resep']}'")
    print(f"  -> Kategori Ditebak: {hasil['Prediksi_Kategori']} (Confidence: {hasil['Tingkat_Keyakinan']})")
    print(f"  -> Kandidat Probabilitas: {hasil['Top_3_Kandidat']}")

## Bagian 7: [TUJUAN 3] Modul Diet Personal (Aturan Deterministik If-Else, BUKAN ML)
### Keputusan Desain & Integritas Ilmiah (Pilihan A):
Pada rencana awal, modul diet dilatih menggunakan model Decision Tree / Logistic Regression pada data sintetis. Namun setelah dievaluasi:
1. Data sintetis tersebut dibuat dengan aturan deterministik WHO, lalu model ML dilatih untuk meniru aturan itu kembali (**masalah penalaran sirkuler**).
2. Di lingkungan produksi/aplikasi medis dan gizi, **aturan deterministik if-else jauh lebih aman, transparan, cepat, dan 100% konsisten**.
3. **Skor perkalian $\text{BMI} \times \text{TDEE}$ dihapus** karena tidak memiliki landasan medis gizi dan tidak digunakan untuk keputusan apapun.

### Standar Medis & Gizi yang Diterapkan:
1. **BMI (Body Mass Index - WHO):**
   $$\text{BMI} = \frac{\text{berat\_kg}}{(\text{tinggi\_m})^2}$$
   - $\text{BMI} < 18.5$: *Underweight* $\rightarrow$ **Bulking (Surplus)** (Rasio kalori per makan: 45%)
   - $18.5 \le \text{BMI} < 25.0$: *Normal* $\rightarrow$ **Maintenance (Seimbang)** (Rasio kalori per makan: 35%)
   - $25.0 \le \text{BMI} < 30.0$: *Overweight* $\rightarrow$ **Diet Ringan (Defisit Ringan)** (Rasio kalori per makan: 30%)
   - $\text{BMI} \ge 30.0$: *Obesitas* $\rightarrow$ **Diet Ketat (Defisit Besar)** (Rasio kalori per makan: 25%)
2. **BMR (Basal Metabolic Rate - Rumus Standar Mifflin-St Jeor):**
   - Pria: $10 \times \text{berat} + 6.25 \times \text{tinggi} - 5 \times \text{umur} + 5$
   - Wanita: $10 \times \text{berat} + 6.25 \times \text{tinggi} - 5 \times \text{umur} - 161$
3. **TDEE (Total Daily Energy Expenditure):**
   - Sedentary: $1.20$, Ringan: $1.375$, Sedang: $1.55$, Berat/Aktif: $1.725$
4. **Satu Aturan Kalori:**
   $$\text{Batas Kalori Per Makan} = \text{round}(\text{TDEE} \times \text{Rasio})$$

In [ ]:
# ==============================================================================
# MODUL DIET PERSONAL: ATURAN DETERMINISTIK IF-ELSE (WHO & MIFFLIN-ST JEOR)
# ==============================================================================

def hitung_bmi(berat_kg, tinggi_cm):
    """Menghitung BMI (Body Mass Index)."""
    tinggi_m = tinggi_cm / 100.0
    return round(berat_kg / (tinggi_m ** 2), 2)

def hitung_bmr(jenis_kelamin, berat_kg, tinggi_cm, umur_thn):
    """Menghitung BMR menggunakan rumus Mifflin-St Jeor."""
    jk = str(jenis_kelamin).strip().lower()
    pria = jk.startswith('l') or 'pria' in jk
    dasar = 10 * berat_kg + 6.25 * tinggi_cm - 5 * umur_thn
    return dasar + 5 if pria else dasar - 161

FAKTOR_AKTIVITAS = {
    'sedentary': 1.2,   # Jarang bergerak / tidak olahraga
    'ringan': 1.375,    # Olahraga ringan 1-3 hari/minggu
    'sedang': 1.55,     # Olahraga sedang 3-5 hari/minggu
    'aktif': 1.725,     # Olahraga berat 6-7 hari/minggu
}

def hitung_tdee(bmr, level_aktivitas='ringan'):
    """Menghitung kebutuhan kalori harian (TDEE)."""
    return round(bmr * FAKTOR_AKTIVITAS.get(level_aktivitas, 1.375))

def tentukan_kategori_diet(bmi):
    """Ambang batas kategori diet berdasarkan pedoman WHO."""
    if bmi >= 30.0:
        return 'Diet Ketat (Defisit Besar)'
    elif bmi >= 25.0:
        return 'Diet Ringan (Defisit Ringan)'
    elif bmi >= 18.5:
        return 'Maintenance (Seimbang)'
    else:
        return 'Bulking (Surplus)'

RASIO_KALORI_PER_MAKAN = {
    'Diet Ketat (Defisit Besar)': 0.25,
    'Diet Ringan (Defisit Ringan)': 0.30,
    'Maintenance (Seimbang)': 0.35,
    'Bulking (Surplus)': 0.45,
}

def hitung_batas_kalori_per_makan(tdee, kategori_diet):
    """Menghitung alokasi batas kalori per satu porsi makan."""
    return round(tdee * RASIO_KALORI_PER_MAKAN.get(kategori_diet, 0.35))

def tentukan_profil_diet_user(jenis_kelamin, berat_kg, tinggi_cm, umur_thn, level_aktivitas='ringan'):
    """Menghitung profil diet lengkap pengguna (Non-ML, 100% deterministik)."""
    bmi = hitung_bmi(berat_kg, tinggi_cm)
    bmr = hitung_bmr(jenis_kelamin, berat_kg, tinggi_cm, umur_thn)
    tdee = hitung_tdee(bmr, level_aktivitas)
    kategori = tentukan_kategori_diet(bmi)
    
    if bmi >= 30: status_bmi = 'Obesitas'
    elif bmi >= 25: status_bmi = 'Overweight'
    elif bmi >= 18.5: status_bmi = 'Normal'
    else: status_bmi = 'Underweight'
    
    return {
        'BMI': bmi,
        'Status_BMI': status_bmi,
        'BMR': round(bmr),
        'TDEE_Harian': tdee,
        'Kategori_Diet': kategori,
        'Rasio_Per_Makan': f"{int(RASIO_KALORI_PER_MAKAN[kategori]*100)}%",
        'Batas_Kalori_Per_Makan': hitung_batas_kalori_per_makan(tdee, kategori)
    }

print("Modul Diet Personal If-Else berhasil diinisialisasi.")

### 7.1 Fitur Aplikasi: Profil Personal Pengguna & Aturan Satu Kalori
Tabel di bawah ini menguji profil diet personal pengguna untuk berbagai skenario berat badan dan gaya hidup:

In [ ]:
daftar_profil_uji = [
    ('Pria', 85, 170, 28, 'sedentary', 'Pria Overweight Sedentary'),
    ('Pria', 68, 172, 23, 'sedang', 'Pria Normal Aktif'),
    ('Wanita', 43, 160, 21, 'ringan', 'Wanita Underweight Ringan'),
    ('Wanita', 80, 155, 35, 'ringan', 'Wanita Obesitas Ringan'),
]

hasil_profil = []
for jk, bb, tb, u, akt, ket in daftar_profil_uji:
    p = tentukan_profil_diet_user(jk, bb, tb, u, akt)
    p['Skenario'] = ket
    hasil_profil.append(p)

df_profil_uji = pd.DataFrame(hasil_profil)[[
    'Skenario', 'BMI', 'Status_BMI', 'BMR', 'TDEE_Harian', 'Kategori_Diet', 'Rasio_Per_Makan', 'Batas_Kalori_Per_Makan'
]]
display(df_profil_uji)

## Bagian 8: SmartCook Unified Engine (Integrasi Utuh 3 Fitur)
Fungsi `smartcook_rekomendasi_komprehensif` menyatukan seluruh modul menjadi satu antarmuka terpadu:
1. **Rekomendasi Berbasis Bahan:** Pencocokan kemiripan bahan dapur via TF-IDF Cosine (token nama bahan utuh).
2. **Filter Kategori Resep:** Memilih jenis protein/bahan utama yang diinginkan (ayam, sapi, ikan, dsb.).
3. **Filter Diet Personal (Kalori per Porsi):** Memastikan resep yang disajikan memenuhi kriteria:
   $$\text{kalori\_per\_porsi} \le \text{Batas Kalori Per Makan}$$
   *(Default: 600 kkal bila pengguna belum melengkapi profil fisik).*

In [ ]:
def smartcook_rekomendasi_komprehensif(
    bahan_dimiliki,
    profil_user=None,
    filter_kategori=None,
    batas_kalori_override=None,
    top_k=5
):
    """
    Mesin Rekomendasi Terpadu SmartCook:
    Menggabungkan Rekomendasi Kemiripan Bahan, Filter Kategori, dan Filter Diet Kalori per Porsi.
    """
    if batas_kalori_override is not None:
        batas_kalori = batas_kalori_override
    elif profil_user is not None:
        batas_kalori = profil_user.get('Batas_Kalori_Per_Makan', 600)
    else:
        batas_kalori = 600
        
    kandidat = rekomendasikan_resep_berbasis_bahan(
        bahan_pengguna=bahan_dimiliki,
        top_k=top_k * 4,
        filter_kategori=filter_kategori
    )
    
    if kandidat.empty:
        return pd.DataFrame()
        
    kandidat['Memenuhi_Diet'] = kandidat['Kalori_Per_Porsi_kkal'] <= batas_kalori
    resep_lolos = kandidat[kandidat['Memenuhi_Diet']].copy()
    if len(resep_lolos) < top_k:
        resep_lolos = kandidat.copy()
        
    resep_lolos = resep_lolos.head(top_k).reset_index(drop=True)
    resep_lolos['Peringkat'] = resep_lolos.index + 1
    resep_lolos['Batas_Kalori_User'] = batas_kalori
    
    return resep_lolos

## Bagian 8.1: Interactive Testing Playground (Uji Coba Input Bahan & Profil Pengguna)
Gunakan sel di bawah ini untuk menguji secara interaktif integrasi sistem rekomendasi dengan berbagai input bahan kulkas dan profil fisik.

In [ ]:
def cetak_kartu_rekomendasi(hasil_df, profil_user=None):
    """Mencetak hasil rekomendasi resep dalam format kartu informasi yang rapi."""
    if hasil_df.empty:
        print("\n[INFO] Tidak ditemukan resep yang memenuhi kriteria.")
        return
        
    print("=" * 75)
    if profil_user:
        print(f"PROFIL USER: {profil_user['Status_BMI']} (BMI: {profil_user['BMI']}) | TDEE: {profil_user['TDEE_Harian']} kkal | Target: {profil_user['Kategori_Diet']}")
        print(f"Batas Kalori Per Makan: {profil_user['Batas_Kalori_Per_Makan']} kkal")
    else:
        print("PROFIL USER: Umum (Batas Kalori Per Makan Default: 600 kkal)")
    print("=" * 75)
    
    for _, row in hasil_df.iterrows():
        status_diet = "SESUAI DIET" if row['Memenuhi_Diet'] else "MELEBIHI BATAS DIET"
        print(f"\n#{row['Peringkat']} [{row['Kategori'].upper()}] {row['Judul']}")
        print(f"   Kemiripan Kosinus : {row['Skor_Cosine']} | Persentase Cocok: {row['Persen_Cocok']}")
        print(f"   Nutrisi Porsi     : {row['Kalori_Per_Porsi_kkal']} kkal/porsi (Est. {row['Estimasi_Porsi']} porsi, Total {row['Kalori_Total_kkal']} kkal / {row['Berat_Total_g']}g)")
        print(f"   Status Diet       : [{status_diet}] (Batas: {row['Batas_Kalori_User']} kkal)")
        print(f"   Bahan Dimiliki    : {', '.join(row['Bahan_Dimiliki']) if row['Bahan_Dimiliki'] else '-'}")
        print(f"   Bahan Kurang      : {', '.join(row['Bahan_Kurang']) if row['Bahan_Kurang'] else '-'}")
        print("-" * 75)

In [ ]:
# ==============================================================================
# SILAKAN UBAH PARAMETER DI BAWAH INI UNTUK MENGETES SISTEM:
# ==============================================================================

# 1. Bahan di Kulkas / Dapur:
BAHAN_KULKAS = "daging ayam, cabai rawit, bawang putih, tomat, garam"

# 2. Profil Fisik Pengguna:
PROFIL_PENGGUNA = tentukan_profil_diet_user(
    jenis_kelamin='Pria',
    berat_kg=82,
    tinggi_cm=170,
    umur_thn=25,
    level_aktivitas='ringan'
)

# 3. Filter Kategori Pilihan (Opsional, isi None jika bebas):
FILTER_KAT = 'ayam'

# Jalankan Rekomendasi Terpadu
hasil_playground = smartcook_rekomendasi_komprehensif(
    bahan_dimiliki=BAHAN_KULKAS,
    profil_user=PROFIL_PENGGUNA,
    filter_kategori=FILTER_KAT,
    top_k=3
)

cetak_kartu_rekomendasi(hasil_playground, PROFIL_PENGGUNA)

## Bagian 9: Panduan Deployment, Ekspor Model, & Sinkronisasi Database
### Arsitektur Integrasi SmartCook:
1. **Database PostgreSQL Supabase (Migrasi 007):**
   - Menggunakan tabel `ingredients` dan relasi `recipe_ingredients`.
   - Bobot IDF dan norma vektor resep dikalkulasi otomatis oleh fungsi `public.refresh_tfidf()` di database.
   - Fungsi RPC `public.match_recipes` mengeksekusi rekomendasi cosine TF-IDF langsung di tingkat query SQL database berkecepatan tinggi.
2. **Backend ML FastAPI (`backend_ml/`):**
   - Hanya melayani endpoint `/predict-category` (Auto-tagging kategori resep).
   - Sel di bawah ini mengekspor model `vec_kategori.joblib` dan `model_kategori_lr.joblib` untuk backend FastAPI.
3. **Aplikasi Flutter:**
   - Modul diet dihitung secara lokal di perangkat klien (`lib/core/utils/nutrition.dart`) menggunakan aturan deterministic if-else.

In [ ]:
import joblib

# Direktori penyimpanan artefak
EXPORT_DIR = 'models_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Simpan model klasifikasi kategori untuk backend FastAPI (/predict-category)
joblib.dump(vec_kategori, os.path.join(EXPORT_DIR, 'vec_kategori.joblib'))
joblib.dump(model_lr, os.path.join(EXPORT_DIR, 'model_kategori_lr.joblib'))
print(f"Model Klasifikasi Kategori tersimpan di: {EXPORT_DIR}/")

# 2. Ekstraksi tabel bobot IDF kosakata bahan untuk verifikasi Supabase
vocab_dict = m_dev['tfidf'].vocabulary_
tabel_idf = pd.DataFrame({
    'term': list(vocab_dict.keys()),
    'idx': list(vocab_dict.values())
}).sort_values('idx').reset_index(drop=True)
tabel_idf['idf'] = m_dev['tfidf'].idf_[tabel_idf['idx'].values]

tabel_idf.to_csv(os.path.join(EXPORT_DIR, 'idf_bahan_reference.csv'), index=False)
print(f"Total bahan terdaftar: {len(tabel_idf):,} entitas bahan.")
print(f"Tabel referensi IDF tersimpan di: {EXPORT_DIR}/idf_bahan_reference.csv")
print("\nContoh 10 Bahan dengan Bobot IDF Terendah (Bahan Paling Umum):\n", tabel_idf.head(10)[['term', 'idf']].to_string(index=False))
print("\nContoh 10 Bahan dengan Bobot IDF Tertinggi (Bahan Paling Spesifik):\n", tabel_idf.tail(10)[['term', 'idf']].to_string(index=False))

## Bagian 10: Rangkuman Hasil Eksperimen untuk Laporan PBL

| Komponen Fitur | Algoritma yang Digunakan | Hasil Evaluasi pada Data Uji (20% Unseen) | Peran di Arsitektur SmartCook |
|---|---|---|---|
| **Rekomendasi Berbasis Bahan** | Content-Based Filtering (TF-IDF + Cosine Similarity, Token Nama Bahan Utuh) | **Hit@1 = 43,8%**, **Hit@5 = 66,0%**, **Hit@10 = 74,6%**, **MRR = 0,527** (Mengungguli word n-gram dan baseline) | Dijalankan langsung di query database Supabase RPC `match_recipes` (Migrasi 007) |
| **Klasifikasi Kategori Resep** | TF-IDF + Logistic Regression (C=5.0) | **Akurasi = 93,2%**, **Precision = 93,1%**, **Recall = 93,0%**, **F1-Score = 93,0%** | Dijalankan di backend FastAPI (`/predict-category`) untuk auto-tagging resep baru |
| **Filter Diet Personal** | Deterministik If-Else (WHO BMI, Mifflin-St Jeor BMR, TDEE) | **100% Konsisten, Reliabel, Non-Sirkuler** (Menghilangkan ketergantungan model ML sintetis) | Dijalankan lokal di aplikasi Flutter (`nutrition.dart`) dan filter kolom `kkal_porsi` |

In [ ]:
ringkasan_pbl = pd.DataFrame([
    {
        'Komponen': 'Rekomendasi Berbasis Bahan',
        'Pendekatan': 'TF-IDF + Cosine (Bahan Utuh)',
        'Metrik Kunci': 'Hit@5: 66.0% | MRR: 0.527',
        'Status Integrasi': 'Selesai (SQL Supabase 007 & Scikit-Learn Ekuivalen)'
    },
    {
        'Komponen': 'Klasifikasi Kategori Resep',
        'Pendekatan': 'TF-IDF + Logistic Regression',
        'Metrik Kunci': 'Akurasi: 93.2% | F1-Score: 93.0%',
        'Status Integrasi': 'Selesai (Model diekspor ke backend_ml/)'
    },
    {
        'Komponen': 'Filter Diet Personal',
        'Pendekatan': 'If-Else Deterministik (WHO & Mifflin-St Jeor)',
        'Metrik Kunci': '100% Valid & Non-Sirkuler',
        'Status Integrasi': 'Selesai (Implementasi Dart & DB kkal_porsi)'
    }
]).set_index('Komponen')

print("=" * 85)
print("RINGKASAN AKHIR PIPELINE MACHINE LEARNING & ANALITIK SMARTCOOK (PBL)")
print("=" * 85)
display(ringkasan_pbl)
print("\n[SUKSES] Seluruh tahapan pipeline dan eksperimen berhasil diselesaikan dengan baik.")